In [ ]:
# Recipe data preprocessing - Minkyung Park (202435338)
# Inputs: RAW_recipes.csv, RAW_interactions.csv, recipes.csv, reviews.csv.
# Set data_path to the folder containing the source CSV files, then run cells in order.
# Final output: data_path / "final_split". Requires pandas and IPython.
# API documentation: https://pandas.pydata.org/docs/reference/index.html
# Python parsing: https://docs.python.org/3/library/ast.html
from pathlib import Path
import pandas as pd
from IPython.display import display

# Set the source CSV directory for Colab or a local environment.
data_path = Path("/content") if Path("/content").is_dir() else Path.cwd()
required_files = ["RAW_recipes.csv", "RAW_interactions.csv", "recipes.csv", "reviews.csv"]
for filename in required_files:
    if not (data_path / filename).is_file():
        raise FileNotFoundError(f"Prepare the source file: {data_path / filename}")

In [ ]:
# Source columns missing values and rating distributions
# Read only the required columns.
raw_recipes = pd.read_csv(
    data_path / "RAW_recipes.csv",
    usecols=["id", "name", "ingredients", "tags"]
)

raw_interactions = pd.read_csv(
    data_path / "RAW_interactions.csv",
    usecols=["user_id", "recipe_id", "rating", "date"]
)

recipes = pd.read_csv(
    data_path / "recipes.csv",
    usecols=["RecipeId", "Name", "RecipeIngredientParts", "Keywords"]
)

reviews = pd.read_csv(
    data_path / "reviews.csv",
    usecols=["AuthorId", "RecipeId", "Rating", "DateSubmitted"]
)

# Inspect row counts and missing values.
tables = {
    "RAW_recipes": raw_recipes,
    "RAW_interactions": raw_interactions,
    "recipes": recipes,
    "reviews": reviews
}

for name, df in tables.items():
    print(f"\n===== {name}: {len(df):,} rows =====")
    print("Missing values by column:")
    print(df.isna().sum().to_string())

# Count records for each rating.
print("\n===== Dataset 1 rating distribution =====")
print(raw_interactions["rating"].value_counts(dropna=False).sort_index())

print("\n===== Dataset 2 rating distribution =====")
print(reviews["Rating"].value_counts(dropna=False).sort_index())

# Inspect the original ingredient and keyword formats.
print("\n===== Dataset 2 ingredient and keyword examples =====")
print("Ingredients:", recipes["RecipeIngredientParts"].iloc[0])
print("Keywords:", recipes["Keywords"].iloc[0])

In [ ]:
# Within dataset and cross dataset duplicate checks
# Check duplicate IDs and user-recipe pairs within each dataset.
print("===== Duplicate recipe IDs =====")
print("Dataset 1:", raw_recipes["id"].duplicated().sum())
print("Dataset 2:", recipes["RecipeId"].duplicated().sum())

print("\n===== Duplicate user-recipe pairs =====")
print(
    "Dataset 1:",
    raw_interactions.duplicated(["user_id", "recipe_id"]).sum()
)
print(
    "Dataset 2:",
    reviews.duplicated(["AuthorId", "RecipeId"]).sum()
)

# Extract IDs and align column names for comparison.
train_pairs = (
    raw_interactions[["user_id", "recipe_id"]]
    .drop_duplicates()
)

test_pairs = (
    reviews[["AuthorId", "RecipeId"]]
    .rename(columns={
        "AuthorId": "user_id",
        "RecipeId": "recipe_id"
    })
    .drop_duplicates()
)

# Find user-recipe pairs shared by both datasets.
overlap = train_pairs.merge(
    test_pairs,
    on=["user_id", "recipe_id"],
    how="inner"
)

print("\n===== Cross-dataset overlap =====")
print(f"Dataset 1 unique user-recipe pairs: {len(train_pairs):,} records")
print(f"Dataset 2 unique user-recipe pairs: {len(test_pairs):,} records")
print(f"Shared pairs: {len(overlap):,} records")
print(f"Overlap as a fraction of Dataset 2 pairs: {len(overlap) / len(test_pairs):.2%}")

# Inspect coverage of known users and recipes.
known_users = test_pairs["user_id"].isin(train_pairs["user_id"])
known_recipes = test_pairs["recipe_id"].isin(raw_recipes["id"])

print("\n===== Dataset 2 ID coverage =====")
print(f"Fraction of Dataset 2 users present in Dataset 1 ratings: {known_users.mean():.2%}")
print(f"Fraction of Dataset 2 recipes present in Dataset 1 metadata: {known_recipes.mean():.2%}")

# Compare recipe names associated with shared recipe IDs.
recipe_check = (
    raw_recipes[["id", "name"]]
    .merge(
        recipes[["RecipeId", "Name"]],
        left_on="id",
        right_on="RecipeId",
        how="inner"
    )
)

print(f"\nShared recipe IDs: {len(recipe_check):,} records")
display(recipe_check[["id", "name", "Name"]].head(10))

In [ ]:
# Remove shared pairs and inspect the common evaluation scope
# Align the review column names.
test_candidates = reviews.rename(columns={
    "AuthorId": "user_id",
    "RecipeId": "recipe_id",
    "Rating": "rating",
    "DateSubmitted": "date"
}).copy()

# Remove user-recipe pairs already present in Dataset 1.
test_candidates = test_candidates.merge(
    train_pairs,
    on=["user_id", "recipe_id"],
    how="left",
    indicator=True
)

test_candidates = (
    test_candidates.loc[test_candidates["_merge"] == "left_only"]
    .drop(columns="_merge")
    .copy()
)

# Check whether each user and recipe appears in the rating history.
known_user = test_candidates["user_id"].isin(
    raw_interactions["user_id"]
)

known_recipe = test_candidates["recipe_id"].isin(
    raw_interactions["recipe_id"]
)

has_recipe_info = test_candidates["recipe_id"].isin(
    raw_recipes["id"]
)

# Inspect a common test scope for the two models.
test_common = test_candidates.loc[
    known_user & known_recipe & has_recipe_info
].copy()

print("===== Dataset 2 candidates after overlap removal =====")
print(f"Total: {len(test_candidates):,} records")
print(f"Records from users absent from Dataset 1 ratings: {(~known_user).sum():,} records")
print(f"Records for recipes absent from Dataset 1 ratings: {(~known_recipe).sum():,} records")

print("\n===== Preliminary common evaluation scope =====")
print(f"Rating records: {len(test_common):,} records")
print(f"Users: {test_common['user_id'].nunique():,} users")
print(f"Recipe: {test_common['recipe_id'].nunique():,} records")

print("\nRatings in the preliminary common scope:")
print(
    test_common["rating"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nTest record counts per user:")
print(
    test_common.groupby("user_id")
    .size()
    .describe()
)

In [ ]:
# Filter ratings from 1 to 5 and retain known IDs
# 1. Retain ratings from 1 to 5 in Dataset 1.
# Retain recipes included in the metadata table.
train_clean = raw_interactions.loc[
    raw_interactions["rating"].between(1, 5)
    & raw_interactions["recipe_id"].isin(raw_recipes["id"])
].copy()

# 2. From the non-overlapping Dataset 2 records,
# retain ratings from 1 to 5.
test_rated = test_candidates.loc[
    test_candidates["rating"].between(1, 5)
].copy()

# 3. Retain only users and recipes present in the cleaned Dataset 1 ratings.
test_clean = test_rated.loc[
    test_rated["user_id"].isin(train_clean["user_id"])
    & test_rated["recipe_id"].isin(train_clean["recipe_id"])
].copy()

# 4. Recheck that cleaned training and test pairs do not overlap.
overlap_check = train_clean[["user_id", "recipe_id"]].merge(
    test_clean[["user_id", "recipe_id"]],
    on=["user_id", "recipe_id"],
    how="inner"
)

assert len(overlap_check) == 0, "Training-test overlap detected!"

print("===== Cleaned Dataset 1 ratings before the validation split =====")
print(f"Rating records: {len(train_clean):,} records")
print(f"Users: {train_clean['user_id'].nunique():,} users")
print(f"Recipe: {train_clean['recipe_id'].nunique():,} records")

print("\n===== Cleaned Test ratings =====")
print(f"Rating records: {len(test_clean):,} records")
print(f"Users: {test_clean['user_id'].nunique():,} users")
print(f"Recipe: {test_clean['recipe_id'].nunique():,} records")

print("\nTest rating distribution:")
print(test_clean["rating"].value_counts().sort_index())

print(f"\nTraining-test overlapping pairs: {len(overlap_check)} records")

In [ ]:
# Prepare ingredient and tag text
import ast
import re

# Parse a list stored as a string.
# Record parsing failures for inspection.
parse_errors = []

def parse_list(value, recipe_id, column):
    # Keep recipe_id and column to identify any malformed source value.
    try:
        result = ast.literal_eval(value)
        if not isinstance(result, list):
            raise ValueError("Expected a list")
        if not all(isinstance(item, str) for item in result):
            raise ValueError("Expected string elements")
        return result
    except (ValueError, SyntaxError, TypeError):
        parse_errors.append((recipe_id, column))
        return []

def normalize_items(items):
    # Preserve multiword ingredients or tags as underscore-separated tokens.
    tokens = []

    for item in items:
        text = item.lower().strip()
        # Replace separators such as spaces and hyphens with underscores.
        text = re.sub(r"[\W_]+", "_", text).strip("_")
        if text:
            tokens.append(text)

    return " ".join(tokens)

# Select recipes appearing in the cleaned Dataset 1 ratings.
recipe_features = (
    raw_recipes.loc[
        raw_recipes["id"].isin(train_clean["recipe_id"])
    ]
    .rename(columns={"id": "recipe_id"})
    .copy()
)

recipe_features["ingredients_list"] = [
    parse_list(value, recipe_id, "ingredients")
    for recipe_id, value in zip(
        recipe_features["recipe_id"],
        recipe_features["ingredients"]
    )
]

recipe_features["tags_list"] = [
    parse_list(value, recipe_id, "tags")
    for recipe_id, value in zip(
        recipe_features["recipe_id"],
        recipe_features["tags"]
    )
]

recipe_features["ingredients_text"] = (
    recipe_features["ingredients_list"].map(normalize_items)
)

recipe_features["tags_text"] = (
    recipe_features["tags_list"].map(normalize_items)
)

recipe_features["content_text"] = (
    recipe_features["ingredients_text"]
    + " "
    + recipe_features["tags_text"]
).str.strip()

print(f"Recipe count: {len(recipe_features):,} records")
print(f"List parsing failures: {len(parse_errors):,} records")
print(
    "Recipes with empty ingredient and tag text:",
    recipe_features["content_text"].eq("").sum()
)

if parse_errors:
    print("Parsing failure examples:", parse_errors[:5])

# Print a complete example without DataFrame truncation.
example = recipe_features.iloc[0]
print("\n===== Text conversion example =====")
print("Recipe:", example["name"])
print("Ingredients:", example["ingredients_text"])
print("Tags:", example["tags_text"])

In [ ]:
# Compare ratings and dates for shared pairs
# Compare shared user-recipe rating records.
comparison = raw_interactions.merge(
    reviews.rename(columns={
        "AuthorId": "user_id",
        "RecipeId": "recipe_id",
        "Rating": "other_rating",
        "DateSubmitted": "other_date"
    }),
    on=["user_id", "recipe_id"],
    how="inner"
)

# Normalize timestamps to UTC calendar dates before comparison.
comparison["date_a"] = pd.to_datetime(
    comparison["date"], errors="coerce", utc=True
).dt.normalize()

comparison["date_b"] = pd.to_datetime(
    comparison["other_date"], errors="coerce", utc=True
).dt.normalize()

same_rating = comparison["rating"].eq(comparison["other_rating"])

valid_dates = (
    comparison["date_a"].notna()
    & comparison["date_b"].notna()
)

same_date = comparison["date_a"].eq(comparison["date_b"])

print(f"Shared rating records: {len(comparison):,} records")
print(f"Rating agreement: {same_rating.mean():.2%}")
print(f"Records with comparable dates: {valid_dates.sum():,} records")

if valid_dates.any():
    print(
        "Date agreement:",
        f"{same_date[valid_dates].mean():.2%}"
    )
    print(
        "Rating and date agreement:",
        f"{(same_rating & same_date)[valid_dates].mean():.2%}"
    )

print("\nExamples with different ratings:")
display(
    comparison.loc[
        ~same_rating,
        ["user_id", "recipe_id", "rating", "other_rating",
         "date", "other_date"]
    ].head(10)
)

In [ ]:
# Compare names for shared recipe IDs
import re
import html
import unicodedata

def normalize_name(value):
    # Normalize names only for cross-source identity checks.
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = unicodedata.normalize("NFKC", text).casefold()
    return re.sub(r"[\W_]+", "", text)

recipe_check["name_a"] = recipe_check["name"].map(normalize_name)
recipe_check["name_b"] = recipe_check["Name"].map(normalize_name)

valid_names = (
    recipe_check["name_a"].ne("")
    & recipe_check["name_b"].ne("")
)

same_name = recipe_check["name_a"].eq(recipe_check["name_b"])
different_names = recipe_check.loc[
    valid_names & ~same_name,
    ["id", "name", "Name"]
]

print(f"Shared recipes: {len(recipe_check):,} records")
print(f"Comparable names: {valid_names.sum():,} records")
print(f"Unavailable names: {(~valid_names).sum():,} records")
print(f"Name matches after normalization: {(valid_names & same_name).sum():,} records")
print(f"Name mismatches after normalization: {len(different_names):,} records")

if valid_names.any():
    print(f"Name agreement: {same_name[valid_names].mean():.2%}")

display(different_names.head(20))

In [ ]:
# Validation split and leakage checks
# Randomly sample 10 percent as validation candidates.
# Return unsupported candidates to Train so both models can evaluate known IDs.
validation_candidate = train_clean.sample(
    frac=0.1,
    random_state=42
)

train_model = train_clean.drop(validation_candidate.index).copy()

# Keep candidates whose users and recipes remain in the provisional Train set.
can_validate = (
    validation_candidate["user_id"].isin(train_model["user_id"])
    & validation_candidate["recipe_id"].isin(train_model["recipe_id"])
)

validation = validation_candidate.loc[can_validate].copy()

# Return unsupported validation candidates to Train.
train_model = pd.concat([
    train_model,
    validation_candidate.loc[~can_validate]
]).sort_index()

# Verify that Validation and Test users and recipes occur in final Train.
for name, table in [
    ("Validation", validation),
    ("Test", test_clean)
]:
    assert table["user_id"].isin(train_model["user_id"]).all(), (
        f"{name}: User absent from Train"
    )
    assert table["recipe_id"].isin(train_model["recipe_id"]).all(), (
        f"{name}: Recipe absent from Train"
    )

# Check every pair of splits for shared user-recipe pairs.
for left, right in [
    (train_model, validation),
    (train_model, test_clean),
    (validation, test_clean)
]:
    assert left[["user_id", "recipe_id"]].merge(
        right[["user_id", "recipe_id"]],
        on=["user_id", "recipe_id"]
    ).empty

assert len(train_model) + len(validation) == len(train_clean)

print(f"Final Train: {len(train_model):,} records")
print(f"Validation: {len(validation):,} records")
print(f"Final Test: {len(test_clean):,} records")
print(f"Actual validation fraction: {len(validation) / len(train_clean):.2%}")
print("\nUser and recipe coverage and split overlap checks passed!")

In [ ]:
# Save the final CSV files
from pathlib import Path

final_path = data_path / "final_split"
final_path.mkdir(parents=True, exist_ok=True)

columns = ["user_id", "recipe_id", "rating", "date"]

outputs = {
    "train_ratings.csv": train_model[columns],
    "validation_ratings.csv": validation[columns],
    "test_ratings.csv": test_clean[columns],
    "recipe_features.csv": recipe_features[[
        "recipe_id", "name",
        "ingredients_text", "tags_text", "content_text"
    ]]
}

for filename, table in outputs.items():
    table.to_csv(
        final_path / filename,
        index=False,
        encoding="utf-8-sig"
    )
    print(f"Saved: {filename} ({len(table):,} rows)")

print("\nOutput directory:", final_path)

In [ ]:
# Verify saved row counts and columns
import pandas as pd
from pathlib import Path

final_path = data_path / "final_split"

expected = {
    "train_ratings.csv": (
        987860, ["user_id", "recipe_id", "rating", "date"]
    ),
    "validation_ratings.csv": (
        83660, ["user_id", "recipe_id", "rating", "date"]
    ),
    "test_ratings.csv": (
        4301, ["user_id", "recipe_id", "rating", "date"]
    ),
    "recipe_features.csv": (
        226590,
        ["recipe_id", "name", "ingredients_text",
         "tags_text", "content_text"]
    )
}

for filename, (expected_rows, expected_columns) in expected.items():
    path = final_path / filename
    assert path.is_file(), f"Missing file: {filename}"

    # Read in chunks to limit memory used by file verification.
    rows = 0
    for chunk in pd.read_csv(path, chunksize=100_000):
        assert chunk.columns.tolist() == expected_columns, (
            f"Unexpected columns: {filename}"
        )
        rows += len(chunk)

    assert rows == expected_rows, (
        f"Unexpected row count: {filename} ({rows:,} rows)"
    )
    print(f"Verified: {filename} ({rows:,} rows)")

In [ ]:
# Standardize date format and verify integrity
import pandas as pd
from pathlib import Path
p = data_path / "final_split"

# Match Test date strings to Train using YYYY-MM-DD and save Test as UTF-8.
te = pd.read_csv(p/"test_ratings.csv")
te["date"] = pd.to_datetime(te["date"], utc=True).dt.strftime("%Y-%m-%d")
te.to_csv(p/"test_ratings.csv", index=False, encoding="utf-8")

tr = pd.read_csv(p / "train_ratings.csv")
va = pd.read_csv(p / "validation_ratings.csv")
rf = pd.read_csv(p/"recipe_features.csv")
key = ["user_id", "recipe_id"]

assert (len(tr), len(va), len(te), len(rf)) == (987860, 83660, 4301, 226590)
for d in (tr, va, te):
    assert d["rating"].between(1, 5).all() and not d.duplicated(key).any()
assert tr.merge(va, on=key).empty and tr.merge(te, on=key).empty and va.merge(te, on=key).empty
for d in (va, te):
    assert d["user_id"].isin(tr["user_id"]).all() and d["recipe_id"].isin(tr["recipe_id"]).all()
assert tr["recipe_id"].isin(rf["recipe_id"]).all() and rf["content_text"].notna().all()
print("Missing names:", rf["name"].isna().sum(), "| Date format:", tr["date"].iloc[0], te["date"].iloc[0])
print("All checks passed")

In [ ]:
# Add cooking time
import pandas as pd
from pathlib import Path

base = data_path
feature_path = final_path / "recipe_features.csv"

features = pd.read_csv(feature_path)

times = pd.read_csv(
    base / "RAW_recipes.csv",
    usecols=["id", "minutes"]
).rename(columns={"id": "recipe_id"})

# Remove an existing minutes column before joining to avoid duplicate columns on rerun.
features = features.drop(columns=["minutes"], errors="ignore")

features = features.merge(
    times,
    on="recipe_id",
    how="left",
    validate="one_to_one"
)

assert len(features) == 226590
assert features["minutes"].notna().all()

features.to_csv(feature_path, index=False, encoding="utf-8-sig")

print("Cooking time added!")
print("Rows:", len(features))
print("Columns:", features.columns.tolist())

In [ ]:
# Verify saved cooking time
saved = pd.read_csv(feature_path)

print("Rows:", len(saved))
print("Columns:", saved.columns.tolist())
print("Missing cooking times:", saved["minutes"].isna().sum())
print("File size (MB):", round(feature_path.stat().st_size / 1024**2, 2))

# Compare the in-memory table with the saved CSV reloaded from disk.
pd.testing.assert_frame_equal(
    features,
    saved,
    check_dtype=False
)

print("Saved and in-memory tables match!")

In [ ]:
# Check the common evaluation user condition
train = pd.read_csv(final_path / "train_ratings.csv")
test = pd.read_csv(final_path / "test_ratings.csv")

train_positive_users = set(
    train.loc[train["rating"] >= 4, "user_id"]
)
test_positive_users = set(
    test.loc[test["rating"] >= 4, "user_id"]
)

evaluation_users = train_positive_users & test_positive_users

print("Users with positive Test ratings:", len(test_positive_users))
print("Users with positive ratings in both Train and Test:", len(evaluation_users))
print("Excluded users:", len(test_positive_users - evaluation_users))